In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Main libraries
import numpy as np
import os

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# We will use Polars for data manipulation
import polars as pl

# Casting types from time to time to have a better autocompletion

from models import train_and_explain, ExperimentResults, Species, ModelType, ALL_SPECIES
from analysis import summarize_performance
from config import Ablation, FEATURES_METADATA
from scipy.stats import lognorm
import cmocean

CMAP = cmocean.cm.thermal

# Use caching for various results
if not os.path.exists("./cache"):
    os.makedirs("./cache")

In [ ]:
# Pick grouping strategy: "tree_id" or "plot_id" or None
group_col = "tree_id"
# Pick model type: "gbdt" or "lasso"
model_type: ModelType = "gbdt"
# Ablation: "all", "tree-level-only", "plot-level-only", "no-defoliation", "max-defoliation"
ablation: Ablation = "all"
use_temporal_cv = True

if use_temporal_cv:
    temporal_label = "temp_yes"
else:
    temporal_label = "temp_no"

weight_shap_fimp = False

if weight_shap_fimp:
    weight_shap_label = "weight_shap"
else:
    weight_shap_label = "no_weight_shap"

use_original_space = True

if use_original_space:
    space_label = "org_space"
else:
    space_label = "quantile_space"


setting_identifier = f"{model_type}-{group_col}-{ablation}-{temporal_label}-{weight_shap_label}-{space_label}"

print(setting_identifier)

In [ ]:
# Train and explain models for all species
# Set use_temporal_cv = True, to enable HierarchicalTemporalGroup
all_results: dict[Species, ExperimentResults] = {}

for species in ALL_SPECIES:
    all_results[species] = train_and_explain(
        species,
        model_type=model_type,
        group_by=group_col,
        ablation=ablation,
        use_temporal_cv=use_temporal_cv,
    )

In [ ]:
# Summarize performance as a table
summarize_performance(
    all_results,
    ablation=ablation,
    model_type=model_type,
    group_col=group_col,
    use_temporal_cv=use_temporal_cv,
)

In [ ]:
from sklearn.metrics import r2_score

r2_results = []


def inv_transform(y_quantile, shape, loc, scale):
    y_quantile = np.clip(y_quantile, 1e-10, 1 - 1e-10)
    return lognorm.ppf(y_quantile, shape, loc, scale) - 1


for species, results in all_results.items():
    shape, loc, scale = results.dist_params

    for fold in range(results.num_folds):
        X_train, y_true_train, y_pred_train = results.get_data(fold, split="train")
        X_test, y_true_test, y_pred_test = results.get_data(fold, split="test")

        # Transform to original units for train
        y_true_orig_train = results.get_inverse_transform(
            y_true_train, results.dist_params
        )
        y_pred_orig_train = results.get_inverse_transform(
            y_pred_train, results.dist_params
        )

        # Transform to original units for test
        y_true_orig_test = results.get_inverse_transform(
            y_true_test, results.dist_params
        )
        y_pred_orig_test = results.get_inverse_transform(
            y_pred_test, results.dist_params
        )

        r2_results.append(
            {
                "species": species,
                "fold": fold,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "quantile_r2 (train)": r2_score(y_true_train, y_pred_train),
                "quantile_r2 (test)": r2_score(y_true_test, y_pred_test),
                "growth_r2 (train)": r2_score(y_true_orig_train, y_pred_orig_train),
                "growth_r2 (test)": r2_score(y_true_orig_test, y_pred_orig_test),
            }
        )

df_r2 = pl.DataFrame(r2_results)

In [ ]:
summary = df_r2.group_by("species").agg(
    [
        # Quantile R²
        pl.col("quantile_r2 (test)").mean().alias("quantile_r2_mean"),
        pl.col("quantile_r2 (test)").std().alias("quantile_r2_std"),
        (
            pl.col("quantile_r2 (test)").std()
            / np.sqrt(pl.col("quantile_r2 (test)").count())
        ).alias("quantile_r2_se"),
        # Growth R²
        pl.col("growth_r2 (test)").mean().alias("growth_r2_mean"),
        pl.col("growth_r2 (test)").std().alias("growth_r2_std"),
        (
            pl.col("growth_r2 (test)").std()
            / np.sqrt(pl.col("growth_r2 (test)").count())
        ).alias("growth_r2_se"),
    ]
)

print(
    f"{'Species':<10} | {'Quantile R² (reported)':<22} | {'Growth R² (original units)':<25}|"
)
print("-" * 65)
for row in summary.iter_rows():
    species = row[0]
    q_mean, q_se = row[1], row[3]
    g_mean, g_se = row[4], row[6]
    quantile_str = f"{q_mean:.3f} ± {q_se:.3f}"
    growth_str = f"{g_mean:.3f} ± {g_se:.3f}"
    print(f"{species:<10} | {quantile_str:<22} | {growth_str:<25} |")

In [ ]:
import polars.selectors as cs

if use_original_space:
    feature_importances = pl.from_dicts(
        [
            {
                "species": species,
                "fold": fold,
                **dict(
                    zip(
                        results.features,
                        np.abs(results.get_shap_values_orig_space(fold, "all")).mean(
                            axis=0
                        ),
                    )
                ),
            }
            for species, results in all_results.items()
            for fold in range(5)
        ]
    ).unpivot(
        on=cs.exclude("species", "fold"),  # type: ignore
        index=["species", "fold"],
        variable_name="feature",
        value_name="shap",
    )
else:
    feature_importances = pl.from_dicts(
        [
            {
                "species": species,
                "fold": fold,
                **dict(
                    zip(
                        results.features,
                        np.abs(results.shap_values[fold].values).mean(axis=0),
                    )
                ),
            }
            for species, results in all_results.items()
            for fold in range(5)
        ]
    ).unpivot(
        on=cs.exclude("species", "fold"),  # type: ignore
        index=["species", "fold"],
        variable_name="feature",
        value_name="shap",
    )

In [ ]:
min_importance = 1.0
max_rank = 10

data = (
    feature_importances.with_columns(
        importance=pl.col("shap").mean().over("species", "feature")
    )
    .with_columns(
        rank=pl.col("importance").rank(descending=True, method="dense").over("species"),
    )
    .filter(
        (
            (pl.col("importance").max().over("feature") >= min_importance)
            | (pl.col("rank").min().over("feature") <= max_rank)
        )
    )
    .join(
        pl.from_dicts(
            [
                {"feature": k, "feature_label": v["label"]}
                for k, v in FEATURES_METADATA.items()
            ]
        ),
        on="feature",
        how="left",
    )
    .with_columns(feature=pl.col("feature_label"))
    .drop("feature_label")
    .sort(
        "species", pl.col("importance").mean().over("feature"), descending=[False, True]
    )
)
# Add a summary over all species
data = pl.concat(
    [
        data,
        data.group_by("feature", "fold")
        .agg(
            shap=pl.col("shap").mean(),
            importance=pl.col("importance").mean(),
        )
        .with_columns(species=pl.lit("all species"))
        .with_columns(
            rank=pl.col("importance")
            .rank(descending=True, method="dense")
            .over("species"),
        )
        .select(data.columns),
    ],
    how="vertical_relaxed",
)


with pl.Config() as cfg:
    cfg.set_tbl_formatting("ASCII_FULL")
    cfg.set_tbl_rows(100)

    stats = (
        data.filter(pl.col("species") != "all species")
        .group_by("feature", "species")
        .agg(pl.mean("rank"))
        .sort(by="rank")
        .filter(pl.col("feature") == "Mean defoliation")
    )

    print(stats)

data

In [ ]:
# Plot using seaborn
num_species = len(feature_importances.select("species").unique())
g = sns.catplot(
    data.with_columns((pl.col("shap") * 100).alias("shap_percent")),
    x="shap_percent",
    y="feature",
    hue="species",
    kind="bar",
    palette=sns.color_palette("cmo.thermal", n_colors=num_species + 1),
    height=8,
    aspect=0.6,
)

g._legend.set_title("Species")

# Capitalize legend labels
new_labels = [label.get_text().capitalize() for label in g._legend.texts[0:]]
for label, new_text in zip(g._legend.texts[0:], new_labels):
    label.set_text(new_text)

# Prepend rank to each feature name of the y-axis
ax = g.facet_axis(0, 0)
yticks = ax.get_yticks()
ylabels = ax.get_yticklabels()
new_ylabels = []
for ytick, ylabel in zip(yticks, ylabels):
    feature_name = ylabel.get_text()
    rank = int(
        data.group_by("feature", "species")
        .agg(pl.col("rank").mean())
        .filter(
            (pl.col("feature") == feature_name) & (pl.col("species") == "all species")
        )
        .select("rank")
        .item()
    )
    new_ylabels.append(f"({rank}) {feature_name}")
ax.set_yticklabels(new_ylabels)

if use_original_space:
    plt.xlabel("Feature importance in original space (mean |SHAP value| %)")
else:
    plt.xlabel("Feature importance (mean |SHAP value| %)")
plt.ylabel("Feature")

model_label = "GBDT" if model_type == "gbdt" else "Lasso"
group_label = "tree identifier" if group_col == "tree_id" else "plot identifier"
ablation_label = "all features" if ablation == "all" else "without defoliation"
plt.title(f"Feature importance ({model_label}, {ablation_label})")

fig = plt.gcf()
plt.savefig(
    f"./figures/importance-{setting_identifier}.pdf",
    bbox_inches="tight",
)

In [ ]:
from explain import plot_dependence, PlotType

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if use_original_space:
    ylim = (-0.5, 0.5)
else:
    ylim = (-10, 10)


for species, ax in zip(ALL_SPECIES, axes.flatten()):
    plot_dependence(
        all_results[species],
        feature="dep_n_tot",
        use_original_space=use_original_space,
        alpha=0.2,
        ax=ax,
        xlim=(0, 100),
        ylim=ylim,
    )
    ax.set_xlabel("Total nitrogen deposition [kg N/ha/yr]")

plt.savefig(f"./figures/Dependence-{setting_identifier}-total_nitrogen.pdf")

In [ ]:
from explain import plot_dependence

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if use_original_space:
    ylim = (-0.5, 0.5)
else:
    ylim = (-10, 10)


for species, ax in zip(ALL_SPECIES, axes.flatten()):
    plot_dependence(
        all_results[species],
        feature="dep_s_so4",
        use_original_space=use_original_space,
        alpha=0.2,
        ax=ax,
        xlim=(0, 100),
        ylim=ylim,
    )
    ax.set_xlabel("Dep. sulphate deposition [kg SO4/ha/yr]")

plt.savefig(f"./figures/Dependence-{setting_identifier}-sulphate_deposition.pdf")

In [ ]:
feature = "defoliation_mean"  # Change this to the feature you want to plot

# One plot per species (4 in total)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if use_original_space:
    ylim = (-0.5, 0.5)
else:
    ylim = (-15, 10)

for species, ax in zip(ALL_SPECIES, axes.flatten()):
    ax.grid(True)

    plot_dependence(
        all_results[species],
        feature=feature,
        use_original_space=use_original_space,
        ax=ax,
        # alpha=0.1,
        xlim=(0, 100),
        plot_type=PlotType.LINE,
        linewidth=4.0,
        show_no_effect=False,
        ylim=ylim,
        # color=all_results[species].shap_values[fold][:, "dep_ph"].values
    )

    plot_dependence(
        all_results[species],
        feature="defoliation_max",
        use_original_space=use_original_space,
        ax=ax,
        xlim=(0, 100),
        color="#ff7f0e",
        plot_type=PlotType.LINE,
        linewidth=4.0,
        show_no_effect=False,
        ylim=ylim,
        with_density=False,
        # color=all_results[species].shap_values[fold][:, "dep_ph"].values
    )

    ax.set_title(species.capitalize())
    ax.set_xlabel("Defoliation [%]")

    fig.legend(
        title="Feature",
        labels=[
            "Mean defoliation (μ)",
            "Mean defoliation (95p)",
            "Max defoliation (μ)",
            "Max defoliation (95p)",
        ],
    )

    fig.suptitle("Dependence of growth rate on mean and max defoliation")

plt.tight_layout()
plt.savefig(f"./figures/Dependence-{setting_identifier}-defoliation.pdf")